In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("LogAnalysis")
    .master("spark://master:7077")
    .config("spark.pyspark.python", "/opt/conda/bin/python")
    .getOrCreate()
)

sc = spark.sparkContext

logs = sc.textFile("hdfs://master:9000/input/server.log")

errors = logs.filter(lambda line: "ERROR" in line)
print(f"Tong so loi: {errors.count()}")

error_types = errors.map(lambda line: (line.split()[3], 1))
error_counts = error_types.reduceByKey(lambda a, b: a + b)
result = error_counts.sortBy(lambda item: (-item[1], item[0])).collect()

print("LOG ANALYSIS RESULT")
for error_type, count in result:
    print(f"{error_type}\t{count}")

spark.stop()

26/04/03 07:34:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Tong so loi: 3
LOG ANALYSIS RESULT
DatabaseConnection	2
FileNotFound	1
